### **Importing libraries**

In [118]:
import numpy as np
import pandas as pd
import GEOparse

from scipy.stats import ttest_ind
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import SelectKBest, f_classif

### **Load dataset from GEO.soft file**
Replace the path with your actual file location

In [119]:
import logging

logging.getLogger("GEOparse").setLevel(logging.ERROR)

In [120]:
gse = GEOparse.get_GEO(
    filepath="../data/GSE18123_family.soft"
)

c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")



### **Inspecting the GEO Sample**

The first GEO sample (GSM) is extracted from the downloaded dataset, and its expression matrix is examined. The dataset contains 54,613 probes, with each row representing a probe identifier (ID_REF) and its corresponding normalized expression value (VALUE).

In [121]:
gsm = list(gse.gsms.values())[0]

print(gsm.table.shape)
gsm.table.head()

(54613, 2)


,ID_REF,VALUE
0,1007_s_at,26.65640
1,1053_at,39.00726
2,117_at,296.01724
3,121_at,209.03575
4,1255_g_at,63.82730


### **Extracting Sample Metadata**

The metadata of each GEO sample (GSM) are extracted to build a structured sample information table. For every sample, the diagnosis, gender, age at blood collection, and microarray platform are retrieved from the available metadata and stored in a pandas DataFrame.

The resulting table contains **285 samples** and **5 metadata fields** (`Sample`, `Diagnosis`, `Gender`, `Age`, and `Platform`), providing the clinical and demographic information required for subsequent data preprocessing and analysis.


In [122]:
samples_info = []

for gsm_id, gsm in gse.gsms.items():

    diagnosis = None
    gender = None
    age = None

    for item in gsm.metadata.get("characteristics_ch1", []):

        if item.startswith("diagnosis:"):
            diagnosis = item.replace("diagnosis:", "").strip()

        elif item.startswith("gender:"):
            gender = item.replace("gender:", "").strip()

        elif item.startswith("age at blood drawing"):
            age = int(item.split(":")[-1].strip())

    samples_info.append({
        "Sample": gsm_id,
        "Diagnosis": diagnosis,
        "Gender": gender,
        "Age": age,
        "Platform": gsm.metadata["platform_id"][0]
    })

df_samples = pd.DataFrame(samples_info)

In [123]:
df_samples

,Sample,Diagnosis,Gender,Age,Platform
0,GSM650510,PDD-NOS,male,118,GPL570
1,GSM650512,PDD-NOS,male,79,GPL570
2,GSM650513,AUTISM,male,169,GPL570
3,GSM650514,AUTISM,male,92,GPL570
4,GSM650515,AUTISM,male,78,GPL570
...,...,...,...,...,...
280,GSM650959,CONTROL,male,28,GPL6244
281,GSM650968,CONTROL,male,51,GPL6244
282,GSM650969,CONTROL,male,37,GPL6244
283,GSM650970,CONTROL,male,49,GPL6244


### **Selecting the GPL6244 Platform**

The dataset contains samples generated using two different microarray platforms: **GPL570 (99 samples)** and **GPL6244 (186 samples)**. 
Since **GPL6244** includes a substantially larger number of samples, it was selected for the subsequent analyses. Using the platform with the larger sample size provides greater statistical power and ensures a more comprehensive analysis while avoiding potential technical variability introduced by combining data from different platforms.

In [124]:
df_samples["Platform"].unique()

<StringArray>
['GPL570', 'GPL6244']
Length: 2, dtype: str

In [125]:
df_samples["Platform"].value_counts()

Platform
GPL6244    186
GPL570      99
Name: count, dtype: int64

In [126]:
df_gpl6244 = df_samples[df_samples["Platform"] == "GPL6244"].copy()

df_gpl6244.shape

(186, 5)

In [127]:
df_gpl6244.head(10)

,Sample,Diagnosis,Gender,Age,Platform
99,GSM650655,ASPERGER'S DISORDER,male,156,GPL6244
100,GSM650656,AUTISM,male,168,GPL6244
101,GSM650657,CONTROL,male,228,GPL6244
102,GSM650658,AUTISM,male,60,GPL6244
103,GSM650659,AUTISM,male,36,GPL6244
104,GSM650660,PDD-NOS,male,72,GPL6244
105,GSM650661,PDD-NOS,male,108,GPL6244
106,GSM650662,ASPERGER'S DISORDER,male,120,GPL6244
107,GSM650663,PDD-NOS,male,48,GPL6244
108,GSM650664,PDD-NOS,male,108,GPL6244


### **Diagnostic Grouping Strategy**

The original dataset contained four diagnostic categories: **CONTROL**, **PDD-NOS**, **AUTISM**, and **ASPERGER'S DISORDER**. According to the **DSM-IV** criteria, Autism, Asperger's Disorder, and PDD-NOS were classified as pervasive developmental disorders and represent related neurodevelopmental conditions within the autism spectrum.

In the updated **DSM-5** framework, these previously separate diagnoses were integrated into a single clinical category known as **Autism Spectrum Disorder (ASD)**.

The original dataset contained four diagnostic categories:

| Diagnosis | Number of Samples |
|------------|------------------|
| CONTROL | 82 |
| PDD-NOS | 48 |
| AUTISM | 41 |
| ASPERGER'S DISORDER | 15 |

Since some diagnostic subgroups contained a limited number of samples, especially **Asperger's Disorder**, performing a multi-class classification task could decrease statistical power and increase the possibility of model overfitting. Therefore, all ASD-related diagnoses (**AUTISM**, **PDD-NOS**, and **ASPERGER'S DISORDER**) were combined into one unified ASD class, while **CONTROL** samples were assigned to the healthy control class.

This conversion resulted in a binary classification problem:

- **Control (Label = 0):** Healthy individuals  
- **ASD (Label = 1):** Individuals with autism spectrum-related diagnoses  

This strategy enables a more robust classification model by increasing the number of samples in the disease group and focusing on the primary objective of ASD detection rather than distinguishing between different ASD subtypes.

In [128]:
def convert_label(diagnosis):
    if "CONTROL" in diagnosis:
        return 0
    else:
        return 1


df_gpl6244["Label"] = df_gpl6244["Diagnosis"].apply(convert_label)

df_gpl6244.head()

,Sample,Diagnosis,Gender,Age,Platform,Label
99,GSM650655,ASPERGER'S DISORDER,male,156,GPL6244,1
100,GSM650656,AUTISM,male,168,GPL6244,1
101,GSM650657,CONTROL,male,228,GPL6244,0
102,GSM650658,AUTISM,male,60,GPL6244,1
103,GSM650659,AUTISM,male,36,GPL6244,1


In [129]:
df_gpl6244["Label"].value_counts()

Label
1    104
0     82
Name: count, dtype: int64

In [130]:
df_gpl6244["Diagnosis"].value_counts()

Diagnosis
CONTROL                82
PDD-NOS                48
AUTISM                 41
ASPERGER'S DISORDER    15
Name: count, dtype: int64

### **Gender Distribution Analysis**

The gender distribution of the selected GPL6244 dataset was examined to evaluate the demographic composition of the samples. The dataset contains **128 male samples** and **58 female samples**.

The distribution across diagnostic groups shows that the control group (**Label = 0**) includes **48 males** and **34 females**, while the ASD group (**Label = 1**) contains **80 males** and **24 females**. This analysis highlights the gender imbalance commonly observed in ASD datasets, where males are more frequently represented than females.

In [131]:
print(df_gpl6244['Gender'].value_counts())

Gender
male      128
female     58
Name: count, dtype: int64


In [132]:
print(df_gpl6244.groupby(['Label', 'Gender']).size())

Label  Gender
0      female    34
       male      48
1      female    24
       male      80
dtype: int64


### **Initial Statistical Description**

The dataset contains 186 samples with complete metadata and no missing values.  
The samples were obtained from the GPL6244 platform. The dataset includes 128 male and 58 female participants.  

The age of participants ranged from 24 to 264 months, with a mean age of 96.8 months.  
The median age was 84 months, indicating a wide age distribution among samples.

All samples were generated using the same microarray platform, ensuring consistency across the dataset.

In [133]:
df_gpl6244.isnull().sum()

Sample       0
Diagnosis    0
Gender       0
Age          0
Platform     0
Label        0
dtype: int64

In [134]:
df_gpl6244.describe(include="all")

,Sample,Diagnosis,Gender,Age,Platform,Label
count,186,186,186,186.000000,186,186.00000
unique,186,4,2,NaN,1,NaN
top,GSM650655,CONTROL,male,NaN,GPL6244,NaN
freq,1,82,128,NaN,186,NaN
mean,NaN,NaN,NaN,96.806452,NaN,0.55914
std,NaN,NaN,NaN,52.409460,NaN,0.49783
min,NaN,NaN,NaN,24.000000,NaN,0.00000
25%,NaN,NaN,NaN,51.250000,NaN,0.00000
50%,NaN,NaN,NaN,84.000000,NaN,1.00000
75%,NaN,NaN,NaN,136.500000,NaN,1.00000


### **Data Exploration**

- **Load GEO Dataset** 
- **Inspect Dataset Structure** 
- **Extract Sample Metadata** 
- **Check Platforms** 
- **Select GPL6244 Platform** 
- **Convert Diagnosis to Binary Labels** 
- **Analyze Diagnosis Distribution** 
- **Analyze Gender Distribution** 
- **Missing Value Analysis** 
- **Initial Statistical Description** 


### **Load Platform Annotation**

The platform annotation (GPL6244) was loaded to retrieve the probe annotation table. This table provides the mapping between probe identifiers and their corresponding gene annotations, which is required to convert probe-level expression data into gene-level expression data.


In [135]:
platform = gse.gpls['GPL6244']
type(platform)

platform_table = platform.table

platform_table.head()

,ID,GB_LIST,SPOT_ID,seqname,RANGE_GB,RANGE_STRAND,RANGE_START,RANGE_STOP,total_probes,gene_assignment,mrna_assignment,category
0,7896736,NaN,chr1:53049-54936,chr1,NC_000001.10,+,53049,54936,7,---,NONHSAT060105 // NONCODE // Non-coding transcr...,main
1,7896738,NaN,chr1:63015-63887,chr1,NC_000001.10,+,63015,63887,31,ENST00000328113 // OR4G2P // olfactory recepto...,ENST00000328113 // ENSEMBL // havana:known chr...,main
2,7896740,"NM_001004195,NM_001005240,NM_001005484,BC13684...",chr1:69091-70008,chr1,NC_000001.10,+,69091,70008,24,"NM_001004195 // OR4F4 // olfactory receptor, f...",NM_001004195 // RefSeq // Homo sapiens olfacto...,main
3,7896742,"NR_024437,XM_006711854,XM_006726377,XR_430662,...",chr1:334129-334296,chr1,NC_000001.10,+,334129,334296,6,NR_024437 // LOC728323 // uncharacterized LOC7...,NR_024437 // RefSeq // Homo sapiens uncharacte...,main
4,7896744,"NM_001005221,NM_001005224,NM_001005277,NM_0010...",chr1:367659-368597,chr1,NC_000001.10,+,367659,368597,36,"NM_001005221 // OR4F29 // olfactory receptor, ...",NM_001005221 // RefSeq // Homo sapiens olfacto...,main


In [136]:
platform.table.columns.tolist()
annotation = platform.table[['ID', 'gene_assignment']]
annotation.head()

,ID,gene_assignment
0,7896736,---
1,7896738,ENST00000328113 // OR4G2P // olfactory recepto...
2,7896740,"NM_001004195 // OR4F4 // olfactory receptor, f..."
3,7896742,NR_024437 // LOC728323 // uncharacterized LOC7...
4,7896744,"NM_001005221 // OR4F29 // olfactory receptor, ..."


### **Load Gene Expression Data**

The expression data for a single sample (`GSM650655`) was loaded. Each row represents the expression value measured for a specific probe on the microarray.

In [137]:
sample = gse.gsms['GSM650655'].table

sample.head()

,ID_REF,VALUE
0,7892501,9.67160
1,7892502,122.91175
2,7892503,151.17249
3,7892504,1935.89058
4,7892505,21.14810


### **Map Probes to Gene Annotations**

The expression data was merged with the platform annotation using the probe identifiers. This step associates each probe with its corresponding gene annotation.


In [138]:
merged = sample.merge(
    annotation,
    left_on='ID_REF',
    right_on='ID',
    how='left'
)
merged = merged.drop(columns=['ID'])
merged.head()

,ID_REF,VALUE,gene_assignment
0,7892501,9.67160,---
1,7892502,122.91175,---
2,7892503,151.17249,---
3,7892504,1935.89058,---
4,7892505,21.14810,---


In [139]:
merged = merged[['gene_assignment','VALUE']]

merged.head()

,gene_assignment,VALUE
0,---,9.67160
1,---,122.91175
2,---,151.17249
3,---,1935.89058
4,---,21.14810


In [140]:
gene_expression = merged.groupby('gene_assignment')['VALUE'].mean()
gene_expression = gene_expression[gene_expression.index != '---']

In [141]:
print(annotation.columns)
print(df_gpl6244.columns)

Index(['ID', 'gene_assignment'], dtype='str')
Index(['Sample', 'Diagnosis', 'Gender', 'Age', 'Platform', 'Label'], dtype='str')


### **Extract Gene Symbols from Annotation**

The `gene_assignment` column contains messy strings with multiple fields separated by `//`. This step extracts only the gene symbol (the second field).

For example, a string like `"NM_001123456 // TP53 // protein binding"` is split into:
- `parts[0]` → `"NM_001123456 "` (accession number)
- `parts[1]` → `" TP53 "` (gene symbol) → after `.strip()` → `"TP53"`
- `parts[2]` → `" protein binding"` (description)

In [142]:
def extract_gene(x):
    if pd.isna(x):
        return None
    
    parts = x.split('//')
    
    if len(parts) > 1:
        return parts[1].strip()
    
    return None


annotation['gene_symbol'] = annotation['gene_assignment'].apply(extract_gene)

annotation.head()

,ID,gene_assignment,gene_symbol
0,7896736,---,NaN
1,7896738,ENST00000328113 // OR4G2P // olfactory recepto...,OR4G2P
2,7896740,"NM_001004195 // OR4F4 // olfactory receptor, f...",OR4F4
3,7896742,NR_024437 // LOC728323 // uncharacterized LOC7...,LOC728323
4,7896744,"NM_001005221 // OR4F29 // olfactory receptor, ...",OR4F29


In [143]:
annotation['gene_symbol'] = annotation['gene_assignment'].apply(
    lambda x: x.split('//')[1].strip() if isinstance(x, str) and '//' in x else None
)

print(annotation['gene_symbol'].isna().sum())
print(annotation['gene_symbol'])
print(annotation['gene_symbol'].nunique())
print(annotation.shape)

8004
0              NaN
1           OR4G2P
2            OR4F4
3        LOC728323
4           OR4F29
           ...    
33292          NaN
33293          NaN
33294          NaN
33295          NaN
33296          NaN
Name: gene_symbol, Length: 33297, dtype: str
23307
(33297, 3)


In [144]:
annotation.head()

,ID,gene_assignment,gene_symbol
0,7896736,---,NaN
1,7896738,ENST00000328113 // OR4G2P // olfactory recepto...,OR4G2P
2,7896740,"NM_001004195 // OR4F4 // olfactory receptor, f...",OR4F4
3,7896742,NR_024437 // LOC728323 // uncharacterized LOC7...,LOC728323
4,7896744,"NM_001005221 // OR4F29 // olfactory receptor, ...",OR4F29


### **Build Gene Expression Matrix for GPL6244 Samples**

After extracting gene symbols from the platform annotation, the labels are prepared by extracting the `Sample` and `Label` columns from `df_gpl6244` and setting the sample ID as the index. Then, only samples belonging to the **GPL6244** platform are filtered into a dictionary.

For each of these 186 samples, the probe-level expression table is merged with the annotation table on probe IDs. Rows without a valid `gene_symbol` are dropped, and probes mapping to the same gene are aggregated by **mean expression value**. The resulting gene-level expression Series for each sample is stored in a list.

Finally, all Series are concatenated column-wise and transposed to create a matrix where **rows = samples (186)** and **columns = genes (23,307)**, ready for machine learning.

In [145]:
labels = df_gpl6244[['Sample','Label']]
labels = labels.set_index('Sample')
labels.head()

,Label
Sample,
GSM650655,1
GSM650656,1
GSM650657,0
GSM650658,1
GSM650659,1


In [146]:
samples_gpl6244 = {}

for gsm_id, gsm in gse.gsms.items():
    if gsm.metadata['platform_id'][0] == 'GPL6244':
        samples_gpl6244[gsm_id] = gsm


expression_list = []

for gsm_id, gsm in samples_gpl6244.items():

    sample_table = gsm.table

    merged = sample_table.merge(
        annotation,
        left_on='ID_REF',
        right_on='ID',
        how='left'
    )

    merged = merged.dropna(subset=['gene_symbol'])

    gene_exp = (
        merged
        .groupby('gene_symbol')['VALUE']
        .mean()
    )

    gene_exp.name = gsm_id

    expression_list.append(gene_exp)

In [147]:
expression_matrix = pd.concat(expression_list, axis=1)
X = expression_matrix.T
X.shape

(186, 23307)

In [148]:
X.describe

<bound method NDFrame.describe of gene_symbol       A1BG       A1CF        A2M     A2ML1   A3GALT2     A4GALT  \
GSM650655    192.72419   94.81553  123.33105  37.14929  42.56794  202.27898   
GSM650656    141.01814   89.08488  159.11287  41.21219  40.55923  172.14171   
GSM650657    241.32819  136.09189  142.54469  67.19435  69.66604  253.55777   
GSM650658    213.99331  130.45098  169.91373  56.76774  48.05590  233.28105   
GSM650659    196.09077  103.62578  182.87590  53.33165  44.50891  212.83645   
...                ...        ...        ...       ...       ...        ...   
GSM650959    284.18539  135.63803  132.00820  57.57557  44.64943  401.74306   
GSM650968    264.35724  123.40528  116.54155  69.48947  38.24810  398.55275   
GSM650969    276.65809  118.54840  103.69394  54.31459  41.86199  280.85379   
GSM650970    258.39175  108.53577  125.45106  49.78975  36.93898  312.71197   
GSM650975    245.67953   82.29823  125.90277  53.96127  34.97214  262.76077   

gene_symbol      

In [149]:
X = np.log2(X + 1)

In [150]:
X.describe().loc[['min','max','mean']]

gene_symbol,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
min,6.996531,5.866428,5.951166,4.964292,4.602143,6.713755,4.852637,6.589592,7.498633,1.946237,...,5.001921,5.732231,6.444473,6.850984,7.547167,4.815232,6.712321,8.635949,7.662802,6.973475
max,8.924365,8.272282,7.796371,7.109406,6.380760,9.323144,7.551846,8.514086,9.075956,4.311603,...,8.030429,7.810994,10.024975,8.765774,8.946877,7.009564,8.890795,11.645912,9.377321,9.070206
mean,7.605650,6.659614,7.033484,5.703369,5.499839,7.567443,5.935856,7.660215,8.344520,3.307564,...,6.563954,6.466264,8.059242,7.862958,8.393786,5.776127,8.221707,10.034501,8.653903,8.251489


In [151]:
print(X.columns[:10].tolist())
print(len(X.columns))

['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2', 'A4GALT', 'A4GNT', 'AAAS', 'AACS', 'AADAC']
23307


### **Preprocessing Steps Completed**

So far, the following preprocessing steps have been performed:

- **Loaded raw expression data** from GSE18123 using GEOparse
- **Extracted metadata** (diagnosis, gender, age, platform) for all 285 samples
- **Selected GPL6244 platform** (186 samples) over GPL570 (99 samples)
- **Converted multi-class diagnoses** (CONTROL, AUTISM, PDD-NOS, ASPERGER) into **binary labels** (Control = 0, ASD = 1)
- **Loaded platform annotation** (GPL6244) containing probe-to-gene mappings
- **Extracted gene symbols** from the messy `gene_assignment` column using string splitting
- **Built gene expression matrix** by merging each sample's probe-level data with the annotation, aggregating probes to genes by mean, and concatenating all 186 samples into a matrix of shape **(186 samples × 23,307 genes)**

### **Remove Sex Chromosome Genes**

Sex chromosome genes (Y-linked and X-linked) are removed from the expression matrix to prevent gender-related bias from influencing the classification model. Since the dataset has an imbalanced gender distribution (128 males vs. 58 females), keeping these genes could lead the model to learn gender-specific patterns rather than autism-related biomarkers.


In [152]:
sex_genes = [
    'DDX3Y',
    'USP9Y',
    'UTY',
    'TMSB4Y',
    'KDM5D',
    'EIF1AY',
    'RPS4Y1',
    'RPS4Y2',
    'ZFY',
    'SRY',
    'PRKY',
    'BPY2',
    'DAZ1',
    'DAZ2',
    'DAZ3',
    'DAZ4',
    'CDY1',
    'CDY2',
    'HSFY1',
    'HSFY2'
]

In [153]:
cols_to_drop = [col for col in sex_genes if col in X.columns]

print(f"Sex genes found:{len(cols_to_drop)}")
print(f"List: {cols_to_drop}")

Sex genes found:13
List: ['DDX3Y', 'USP9Y', 'UTY', 'TMSB4Y', 'KDM5D', 'EIF1AY', 'RPS4Y1', 'RPS4Y2', 'ZFY', 'SRY', 'PRKY', 'DAZ4', 'HSFY1']


**Result:** 13 sex chromosome genes were found and removed, reducing the feature set from **23,307** to **23,294** genes.

### **Join Labels with Expression Data**

The filtered expression matrix is joined with the binary labels to create the final dataset for modeling.


In [154]:
df_filtered = X.drop(columns=cols_to_drop)

print(f"Genes before: {X.shape[1]}")
print(f"Genes after: {df_filtered.shape[1]}")

Genes before: 23307
Genes after: 23294


In [155]:
data = df_filtered.join(labels)

In [156]:
print(data.shape)
print(data.columns[:10].tolist())
print(data['Label'].value_counts())

(186, 23295)
['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2', 'A4GALT', 'A4GNT', 'AAAS', 'AACS', 'AADAC']
Label
1    104
0     82
Name: count, dtype: int64


In [157]:
labels = labels.loc[X.index]
y = labels['Label']

In [158]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [159]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

(148, 23307)
(38, 23307)
Label
1    83
0    65
Name: count, dtype: int64
Label
1    21
0    17
Name: count, dtype: int64


In [166]:
gene_variances = X_train.var()

In [167]:
print(gene_variances.describe())

count    23307.000000
mean         0.189346
std          0.269216
min          0.005321
25%          0.091309
50%          0.133472
75%          0.203773
max          8.979898
dtype: float64


In [168]:
print("Max variance:")
print(gene_variances.max())

print("\nTop 20 most variable genes:")
print(gene_variances.sort_values(ascending=False).head(20))

Max variance:
8.979898100975339

Top 20 most variable genes:
gene_symbol
RNU6-274P      8.979898
UTY            8.964371
EIF1AY         8.532955
DDX3Y          8.255383
RPS4Y2         7.861321
SNORD115-25    6.667224
KDM5D          6.610227
RPS4Y1         6.467222
TXLNGY         5.880925
USP9Y          5.554086
SNORD115-41    5.285298
RNU6-1116P     4.439922
OCLN           4.343839
UBTFL1         4.342586
SNORD115-6     4.149674
SNORD115-5     4.132891
RNA5SP198      4.021304
SNORD115-31    4.010073
GPR32P1        3.957635
SNORD115-1     3.855573
dtype: float64


In [169]:
print(gene_variances.quantile([0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

0.25    0.091309
0.50    0.133472
0.75    0.203773
0.90    0.328645
0.95    0.470496
0.99    1.031159
dtype: float64


In [170]:
top_genes = gene_variances.sort_values(
    ascending=False
).head(5000).index

X_train_var = X_train[top_genes]
X_test_var = X_test[top_genes]

print(X_train_var.shape)
print(X_test_var.shape)

(148, 5000)
(38, 5000)


In [171]:
results = []

for gene in X_train_var.columns:
    
    control = X_train_var.loc[y_train == 0, gene]
    asd = X_train_var.loc[y_train == 1, gene]
    
    # Welch's t-test
    t_stat, p_value = ttest_ind(
        control,
        asd,
        equal_var=False
    )
    
    control_mean = control.mean()
    asd_mean = asd.mean()
    
    results.append([
        gene,
        control_mean,
        asd_mean,
        asd_mean - control_mean,
        t_stat,
        p_value
    ])


ttest_results = pd.DataFrame(
    results,
    columns=[
        'gene',
        'control_mean',
        'asd_mean',
        'logFC',
        't_stat',
        'p_value'
    ]
)


ttest_results['FDR'] = multipletests(
    ttest_results['p_value'],
    method='fdr_bh'
)[1]


ttest_results = ttest_results.sort_values(
    'FDR'
)


ttest_results.head(20)

,gene,control_mean,asd_mean,logFC,t_stat,p_value,FDR
2158,TIGD7,6.092733,6.599526,0.506793,-5.626649,1.225770e-07,0.000613
774,CCDC144A,7.098078,7.717249,0.619172,-5.329990,3.954694e-07,0.000989
955,ZNF721,7.556653,8.123215,0.566563,-5.140503,9.035127e-07,0.001506
1156,ESF1,6.107429,6.623526,0.516097,-4.859057,3.187508e-06,0.003188
4168,SCARNA17,11.750530,12.158742,0.408213,-5.008906,3.162026e-06,0.003188
3418,ZNF600,6.897923,7.290234,0.392311,-4.796186,4.520814e-06,0.003229
3227,STARD4,7.119018,7.525045,0.406027,-4.822917,4.463041e-06,0.003229
2836,RBM12B,8.564498,8.984790,0.420292,-4.751317,6.654673e-06,0.004159
4181,RNU6-549P,5.119692,5.466472,0.346779,-4.573376,1.032832e-05,0.004820
3558,ZNF830,7.142393,7.518488,0.376095,-4.550189,1.349567e-05,0.004820


In [172]:
significant_genes = ttest_results[
    ttest_results['FDR'] < 0.05
]

print(significant_genes.shape)

(764, 7)


In [173]:
print("Upregulated:")
print((significant_genes['logFC'] > 0).sum())

print("Downregulated:")
print((significant_genes['logFC'] < 0).sum())

Upregulated:
614
Downregulated:
150


In [174]:
significant_genes.head(20)

,gene,control_mean,asd_mean,logFC,t_stat,p_value,FDR
2158,TIGD7,6.092733,6.599526,0.506793,-5.626649,1.225770e-07,0.000613
774,CCDC144A,7.098078,7.717249,0.619172,-5.329990,3.954694e-07,0.000989
955,ZNF721,7.556653,8.123215,0.566563,-5.140503,9.035127e-07,0.001506
1156,ESF1,6.107429,6.623526,0.516097,-4.859057,3.187508e-06,0.003188
4168,SCARNA17,11.750530,12.158742,0.408213,-5.008906,3.162026e-06,0.003188
3418,ZNF600,6.897923,7.290234,0.392311,-4.796186,4.520814e-06,0.003229
3227,STARD4,7.119018,7.525045,0.406027,-4.822917,4.463041e-06,0.003229
2836,RBM12B,8.564498,8.984790,0.420292,-4.751317,6.654673e-06,0.004159
4181,RNU6-549P,5.119692,5.466472,0.346779,-4.573376,1.032832e-05,0.004820
3558,ZNF830,7.142393,7.518488,0.376095,-4.550189,1.349567e-05,0.004820


In [175]:
significant_genes['FDR'].max()

np.float64(0.049974354227373466)

In [176]:
print("Upregulated:", (significant_genes['logFC'] > 0).sum())
print("Downregulated:", (significant_genes['logFC'] < 0).sum())

Upregulated: 614
Downregulated: 150


In [177]:
selected_genes = significant_genes['gene'].tolist()

X_train_lasso = X_train_var[selected_genes]
X_test_lasso = X_test_var[selected_genes]

print(X_train_lasso.shape)
print(X_test_lasso.shape)

(148, 764)
(38, 764)


In [178]:
selected_genes = significant_genes['gene'].tolist()

X_train_lasso = X_train_var[selected_genes]
X_test_lasso = X_test_var[selected_genes]

print(X_train_lasso.shape)
print(X_test_lasso.shape)

(148, 764)
(38, 764)


In [179]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_lasso)
X_test_scaled = scaler.transform(X_test_lasso)

In [180]:
from sklearn.linear_model import LogisticRegressionCV

lasso = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty='l1',
    solver='liblinear',
    scoring='roc_auc',
    max_iter=5000,
    random_state=42
)

lasso.fit(X_train_scaled, y_train)

c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

,"Cs Cs: int or list of floats, default=10Each of the values in Cs describes the inverse of regularizationstrength. If Cs is as an int, then a grid of Cs values are chosenin a logarithmic scale between 1e-4 and 1e4.Like in support vector machines, smaller values specify strongerregularization.",10
,"l1_ratios l1_ratios: array-like of shape (n_l1_ratios), default=NoneFloats between 0 and 1 passed as Elastic-Net mixing parameter (scaling betweenL1 and L2 penalties). For `l1_ratio = 0` the penalty is an L2 penalty. For`l1_ratio = 1` it is an L1 penalty. For `0 < l1_ratio < 1`, the penalty is acombination of L1 and L2.All the values of the given array-like are tested by cross-validation and theone giving the best prediction score is used... warning:: Certain values of `l1_ratios`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... deprecated:: 1.8 `l1_ratios=None` is deprecated in 1.8 and will raise an error in version 1.10. Default value will change from `None` to `(0.0,)` in version 1.10.",'warn'
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"cv cv: int or cross-validation generator, default=NoneThe default cross-validation generator used is Stratified K-Folds.If an integer is provided, it specifies the number of folds, `n_folds`, used.See the module :mod:`sklearn.model_selection` module for thelist of possible cross-validation objects... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer dual=False whenn_samples > n_features.",False
,"penalty penalty: {'l1', 'l2', 'elasticnet'}, default='l2'Specify the norm of the penalty:- `'l2'`: add a L2 penalty term (used by default);- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l1'
,"scoring scoring: str or callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: :ref:`accuracy ` is used.",'roc_auc'
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' might be slower in :class:`LogisticRegressionCV` because it does not handle warm-starting.- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass settin

In [181]:
coefficients = pd.DataFrame({
    'gene': selected_genes,
    'coefficient': lasso.coef_[0]
})

selected_lasso_genes = coefficients[
    coefficients['coefficient'] != 0
].sort_values(
    'coefficient',
    key=abs,
    ascending=False
)

print(selected_lasso_genes.shape)
selected_lasso_genes.head(20)

(70, 2)


,gene,coefficient
503,RNU6-258P,1.855921
492,ANP32E,-1.757487
619,HMGB2,-1.321065
633,PRDX3,-1.290710
335,NGFRAP1,1.242809
17,RN7SL20P,1.190296
125,GMCL1P1,-1.163565
35,YES1,1.053378
45,RPL23,1.038041
223,ALPP,-1.021616


In [182]:
# گرفتن اسم ژن‌های انتخاب شده توسط LASSO
lasso_genes = selected_lasso_genes['gene'].tolist()

# انتخاب همان ژن‌ها از Train و Test
X_train_svm = X_train_var[lasso_genes]
X_test_svm = X_test_var[lasso_genes]

print(X_train_svm.shape)
print(X_test_svm.shape)

(148, 70)
(38, 70)


In [183]:
from sklearn.preprocessing import StandardScaler

svm_scaler = StandardScaler()

X_train_svm_scaled = svm_scaler.fit_transform(X_train_svm)
X_test_svm_scaled = svm_scaler.transform(X_test_svm)

In [184]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel='rbf',
    probability=True,
    random_state=42
)

svm_model.fit(
    X_train_svm_scaled,
    y_train
)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [185]:
y_pred = svm_model.predict(X_test_svm_scaled)

y_prob = svm_model.predict_proba(
    X_test_svm_scaled
)[:,1]

In [186]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:",
      accuracy_score(y_test, y_pred))

print("Precision:",
      precision_score(y_test, y_pred))

print("Recall:",
      recall_score(y_test, y_pred))

print("F1:",
      f1_score(y_test, y_pred))

print("AUC:",
      roc_auc_score(y_test, y_prob))


print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8157894736842105
Precision: 0.85
Recall: 0.8095238095238095
F1: 0.8292682926829268
AUC: 0.8431372549019608

Confusion Matrix:
[[14  3]
 [ 4 17]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.82      0.80        17
           1       0.85      0.81      0.83        21

    accuracy                           0.82        38
   macro avg       0.81      0.82      0.81        38
weighted avg       0.82      0.82      0.82        38



In [187]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegressionCV
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score


X = X_var.copy()
y = y.copy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

auc_scores = []


for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):

    print(f"Fold {fold}")

    # split fold
    X_train_cv = X.iloc[train_idx]
    X_val_cv = X.iloc[val_idx]

    y_train_cv = y.iloc[train_idx]
    y_val_cv = y.iloc[val_idx]


    # -------------------------
    # 1) Variance filtering
    # -------------------------
    variances = X_train_cv.var()

    top_genes = (
        variances
        .sort_values(ascending=False)
        .head(5000)
        .index
    )

    X_train_cv = X_train_cv[top_genes]
    X_val_cv = X_val_cv[top_genes]


    # -------------------------
    # 2) t-test با f_classif
    # -------------------------
    selector = SelectKBest(
        score_func=f_classif,
        k=800
    )

    X_train_cv = selector.fit_transform(
        X_train_cv,
        y_train_cv
    )

    X_val_cv = selector.transform(
        X_val_cv
    )


    # -------------------------
    # 3) Scaling
    # -------------------------
    scaler = StandardScaler()

    X_train_cv = scaler.fit_transform(
        X_train_cv
    )

    X_val_cv = scaler.transform(
        X_val_cv
    )


    # -------------------------
    # 4) LASSO
    # -------------------------
    lasso = LogisticRegressionCV(
        Cs=10,
        cv=5,
        penalty='l1',
        solver='liblinear',
        max_iter=5000,
        random_state=42
    )

    lasso.fit(
        X_train_cv,
        y_train_cv
    )


    # -------------------------
    # 5) SVM
    # -------------------------
    svm = SVC(
        kernel='rbf',
        probability=True,
        random_state=42
    )

    svm.fit(
        X_train_cv,
        y_train_cv
    )


    # prediction
    y_prob = svm.predict_proba(
        X_val_cv
    )[:,1]


    auc = roc_auc_score(
        y_val_cv,
        y_prob
    )

    auc_scores.append(auc)

    print("AUC:", auc)


print("\nAll AUCs:")
print(auc_scores)

print("\nMean AUC:", np.mean(auc_scores))
print("STD:", np.std(auc_scores))

Fold 1


c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

AUC: 0.7507002801120448
Fold 2


c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

AUC: 0.7232142857142858
Fold 3


...

AUC: 0.7142857142857143
Fold 4


c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

AUC: 0.738095238095238
Fold 5


c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

AUC: 0.7147058823529411

All AUCs:
[0.7507002801120448, 0.7232142857142858, 0.7142857142857143, 0.738095238095238, 0.7147058823529411]

Mean AUC: 0.7282002801120449
STD: 0.01417557227514877


In [188]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV


X = X_var.copy()
y = y.copy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

selected_genes_cv = []


for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):

    print(f"Fold {fold}")

    X_train_cv = X.iloc[train_idx]
    y_train_cv = y.iloc[train_idx]


    # -------------------------
    # 1) Variance filtering
    # -------------------------
    variances = X_train_cv.var()

    top_genes = (
        variances
        .sort_values(ascending=False)
        .head(5000)
        .index
    )

    X_train_cv = X_train_cv[top_genes]


    # -------------------------
    # 2) t-test با F-test
    # -------------------------
    from sklearn.feature_selection import SelectKBest, f_classif

    selector = SelectKBest(
        score_func=f_classif,
        k=800
    )

    X_train_cv = selector.fit_transform(
        X_train_cv,
        y_train_cv
    )


    selected_after_test = top_genes[selector.get_support()]


    # -------------------------
    # 3) Scaling
    # -------------------------
    scaler = StandardScaler()

    X_train_cv = scaler.fit_transform(
        X_train_cv
    )


    # -------------------------
    # 4) LASSO
    # -------------------------
    lasso = LogisticRegressionCV(
        Cs=10,
        cv=5,
        penalty='l1',
        solver='liblinear',
        max_iter=5000,
        random_state=42
    )


    lasso.fit(
        X_train_cv,
        y_train_cv
    )


    # ژن‌هایی که ضریب غیر صفر دارند
    coef = lasso.coef_[0]

    selected = selected_after_test[coef != 0]


    print("Selected genes:", len(selected))


    selected_genes_cv.extend(selected.tolist())

Fold 1


c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

Selected genes: 91
Fold 2


c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

Selected genes: 53
Fold 3


...

Selected genes: 40
Fold 4


c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

Selected genes: 745
Fold 5


c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be si

Selected genes: 121


In [189]:
gene_frequency = (
    pd.Series(selected_genes_cv)
    .value_counts()
)


gene_frequency.head(30)

SNORD3B-1       5
LOC101930405    5
HRASLS2         5
GMCL1P1         5
FTH1P2          5
RNU6-547P       5
ALPP            4
CD36            4
OSBP2           4
HTN1            4
ANKRD30BL       4
RNU2-34P        4
SCARNA7         3
ZNF204P         3
RNU6-258P       3
ZNF594          3
PLA2G7          3
ZNF667-AS1      3
PPA1            3
MCEMP1          3
RNU6-1086P      3
CPSF4L          3
YES1            3
RIOK2           3
DHRS7           3
APAF1           3
RNU6-795P       3
HLA-DRB9        3
RNU6-81P        3
USP53           3
Name: count, dtype: int64

In [190]:
stable_genes = gene_frequency[
    gene_frequency >= 3
].index


stable_table = (
    pd.DataFrame({
        "gene": stable_genes,
        "frequency": gene_frequency[stable_genes].values
    })
    .sort_values(
        "frequency",
        ascending=False
    )
)

stable_table

,gene,frequency
0,SNORD3B-1,5
1,LOC101930405,5
2,HRASLS2,5
3,GMCL1P1,5
4,FTH1P2,5
5,RNU6-547P,5
6,ALPP,4
7,CD36,4
8,OSBP2,4
9,HTN1,4


In [191]:
final_candidates = stable_table.merge(
    ttest_results,
    on="gene",
    how="left"
)

final_candidates.head(30)

,gene,frequency,control_mean,asd_mean,logFC,t_stat,p_value,FDR
0,SNORD3B-1,5,8.471782,9.356176,0.884394,-4.045147,0.000092,0.007882
1,LOC101930405,5,7.933042,7.359249,-0.573793,3.642474,0.000377,0.012099
2,HRASLS2,5,5.049475,4.575419,-0.474056,3.347670,0.001037,0.018061
3,GMCL1P1,5,3.678964,3.236479,-0.442485,3.725887,0.000279,0.011015
4,FTH1P2,5,10.745981,10.424183,-0.321799,3.835591,0.000217,0.010217
5,RNU6-547P,5,4.048994,4.306838,0.257845,-3.304543,0.001219,0.019312
6,ALPP,4,5.802979,5.411043,-0.391936,3.481725,0.000663,0.014765
7,CD36,4,9.374792,9.700781,0.325989,-3.195488,0.001744,0.022250
8,OSBP2,4,7.408694,7.059886,-0.348808,3.240811,0.001558,0.021412
9,HTN1,4,4.031256,3.701244,-0.330013,3.412109,0.000840,0.016036
